In [ ]:
import os
import sys

ENDWITHS = 'page-level'

NOTEBOOK_DIR = os.getcwd()

if not NOTEBOOK_DIR.endswith(ENDWITHS):
    raise ValueError(f"Not in correct dir, expect end with {ENDWITHS}, but got {NOTEBOOK_DIR} instead")

BASE_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', '..', '..'))
print(BASE_DIR)

sys.path.insert(0, os.path.join(BASE_DIR))

In [ ]:
from datasets import load_dataset
from MangaDialougeDatasetCreator import MangaTranslationSample, create_training_samples_from_df

ds = load_dataset("ryo0634/bsd_ja_en")

df = ds['train'].with_format('pandas')[:]

In [ ]:
output_dir = os.path.join(BASE_DIR, 'data', 'translation', 'bsd_ja_en', 'training')
os.makedirs(output_dir, exist_ok=True)

SEED= 42

# Config 1: Full Context (baseline - minimal unknowns, wide context)
CONFIG_FULL_CONTEXT = {
    "name": "full_context",
    "unknown_scene_prob": 0.1,      # 10% scenes unknown
    "unknown_speaker_prob": 0.1,    # 10% speakers unknown
    "context_window": 5,             # Wide context window
    "augmentation_passes": 1,        # Moderate augmentation
    "add_ocr_noise": False,          # No noise
    "ocr_noise_prob": 0.0
}

# Config 2: Partial Context (realistic - moderate unknowns, normal context)
CONFIG_PARTIAL_CONTEXT = {
    "name": "partial_context",
    "unknown_scene_prob": 0.3,      # 30% scenes unknown
    "unknown_speaker_prob": 0.2,    # 20% speakers unknown
    "context_window": 3,             # Normal context window
    "augmentation_passes": 2,        # Higher augmentation
    "add_ocr_noise": True,           # Light OCR noise
    "ocr_noise_prob": 0.1            # 10% chars have noise
}

# Config 3: Minimal Context (robustness - heavy unknowns, narrow context)
CONFIG_MINIMAL_CONTEXT = {
    "name": "minimal_context",
    "unknown_scene_prob": 0.5,      # 50% scenes unknown
    "unknown_speaker_prob": 0.4,    # 40% speakers unknown
    "context_window": 2,             # Narrow context window
    "augmentation_passes": 3,        # Heavy augmentation
    "add_ocr_noise": True,           # Heavy OCR noise
    "ocr_noise_prob": 0.15           # 15% chars have noise
}

# All configs
TRAINING_CONFIGS = [
    CONFIG_FULL_CONTEXT,
    CONFIG_PARTIAL_CONTEXT,
    CONFIG_MINIMAL_CONTEXT
]

In [ ]:
df.head(10)

In [ ]:
for config in TRAINING_CONFIGS:
    output_file = os.path.join(output_dir, f"train_{config['name']}.jsonl")
    create_training_samples_from_df(df, output_file, config, seed=SEED)